In [3]:
import torch
torch.cuda.is_available()

True

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os
import json

In [10]:
# --- 1. CONFIGURATION ---

# Paths to your split dataset
absolute_path = '/mnt/c/Users/shiva/Project/Smart Retail Checkout System/'
train_dir = absolute_path + 'processed_data/train'
test_dir = absolute_path + 'processed_data/val'

# Model parameters
IMG_SIZE = 150
BATCH_SIZE = 32
LEARNING_RATE = 0.001
EPOCHS = 25 # Start with 25, you can increase if needed



In [11]:
# --- 2. DATA LOADING & TRANSFORMATION ---

# Define transformations for the training and testing data
# For training, we apply data augmentation to make the model more robust
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # Standard normalization for pre-trained models, also good for custom ones
])

# For testing, we only do the necessary resizing and normalization
test_transforms = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32), # Resize larger
    transforms.CenterCrop(IMG_SIZE),  # Then crop to the center
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("Loading datasets...")

# Use ImageFolder to load the datasets
# This is a key PyTorch utility that assumes your data is organized in class-named folders
train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
test_dataset = datasets.ImageFolder(test_dir, transform=test_transforms)

# Use DataLoader to create iterable batches of data
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)



Loading datasets...


In [12]:
# --- 3. CREATE AND SAVE THE CLASS REFERENCE DICTIONARY ---

# ImageFolder creates a 'class_to_idx' dictionary. We want the reverse for our application.
idx_to_class = {v: k for k, v in train_dataset.class_to_idx.items()}
num_classes = len(idx_to_class)

# Save this dictionary to a JSON file for later use in your real-time script
with open('class_mapping.json', 'w') as f:
    json.dump(idx_to_class, f)

print(f"Found {num_classes} classes: {train_dataset.class_to_idx}")
print("Class mapping dictionary saved to 'class_mapping.json'")


Found 6 classes: {'Butter cookies': 0, 'Chana Chur': 1, 'Chipotle sauce': 2, 'Punjabi tadka': 3, 'Shahi Dates': 4, 'Spicy Coated Peanuts': 5}
Class mapping dictionary saved to 'class_mapping.json'


In [13]:
# --- 4. DEFINE THE CNN MODEL ---

class ProductCNN(nn.Module):
    def __init__(self, num_classes):
        super(ProductCNN, self).__init__()
        # Feature Extraction Layers
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        
        # Calculate the flattened size dynamically
        # We run a dummy tensor through the feature layers to see the output size
        with torch.no_grad():
            dummy_input = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)
            dummy_output_size = self.features(dummy_input).view(1, -1).size(1)

        # Classifier (Fully Connected) Layers
        self.classifier = nn.Sequential(
            nn.Linear(dummy_output_size, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5), # Dropout for regularization
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1) # Flatten the feature maps
        x = self.classifier(x)
        return x



In [14]:
# --- 5. TRAINING SETUP ---

# Check for GPU availability and set the device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Instantiate the model and move it to the selected device
model = ProductCNN(num_classes).to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)



Using device: cuda:0


In [15]:
# --- 6. THE TRAINING LOOP ---

print("\nStarting model training...")

for epoch in range(EPOCHS):
    # --- Training Phase ---
    model.train() # Set the model to training mode
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for i, (inputs, labels) in enumerate(train_loader):
        # Move inputs and labels to the device (GPU or CPU)
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Track statistics
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_train_loss = running_loss / len(train_loader.dataset)
    epoch_train_acc = correct_train / total_train

    # --- Testing/Validation Phase ---
    model.eval() # Set the model to evaluation mode
    running_loss_test = 0.0
    correct_test = 0
    total_test = 0
    with torch.no_grad(): # No need to calculate gradients during testing
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss_test += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()

    epoch_test_loss = running_loss_test / len(test_loader.dataset)
    epoch_test_acc = correct_test / total_test
    
    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} | "
          f"Test Loss: {epoch_test_loss:.4f} | Test Acc: {epoch_test_acc:.4f}")





Starting model training...
Epoch 1/25 | Train Loss: 2.2488 | Train Acc: 0.1176 | Test Loss: 1.7667 | Test Acc: 0.2857
Epoch 2/25 | Train Loss: 1.7150 | Train Acc: 0.2549 | Test Loss: 1.7023 | Test Acc: 0.2857
Epoch 3/25 | Train Loss: 1.6822 | Train Acc: 0.2745 | Test Loss: 1.5265 | Test Acc: 0.4286
Epoch 4/25 | Train Loss: 1.4261 | Train Acc: 0.5000 | Test Loss: 1.3024 | Test Acc: 0.5000
Epoch 5/25 | Train Loss: 1.1929 | Train Acc: 0.5294 | Test Loss: 1.0170 | Test Acc: 0.5714
Epoch 6/25 | Train Loss: 1.0632 | Train Acc: 0.6078 | Test Loss: 0.9895 | Test Acc: 0.7143
Epoch 7/25 | Train Loss: 0.9535 | Train Acc: 0.5686 | Test Loss: 0.7344 | Test Acc: 0.6429
Epoch 8/25 | Train Loss: 0.8923 | Train Acc: 0.6176 | Test Loss: 0.8675 | Test Acc: 0.7143
Epoch 9/25 | Train Loss: 0.7700 | Train Acc: 0.7647 | Test Loss: 0.6870 | Test Acc: 0.6429
Epoch 10/25 | Train Loss: 0.7028 | Train Acc: 0.7353 | Test Loss: 0.5382 | Test Acc: 0.7143
Epoch 11/25 | Train Loss: 0.6113 | Train Acc: 0.7941 | Test L

In [16]:
# --- 7. SAVE THE TRAINED MODEL ---
model_save_path = 'smart_retail_model.pth'
torch.save(model.state_dict(), model_save_path)

print("\nTraining complete!")
print(f"Model state dictionary saved to '{model_save_path}'")


Training complete!
Model state dictionary saved to 'smart_retail_model.pth'
